In [1]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
torch.manual_seed(42)


In [2]:
ATTACK_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Brute Force\\combined_brute_force.csv"   # change per attack
BENIGN_FILE = "C:\\Users\\johfr\\Dropbox\\Skule\\Data science\\Bachelor\\Dataset\\Benign\\combined_benign.csv"
MODEL_PATH   = "brute_force_ann.pth" 
SAMPLE_CAP  = 50000
RANDOM_SEED = 42

df_attack = pd.read_csv(ATTACK_FILE)
df_attack = df_attack.sample(n=min(len(df_attack), SAMPLE_CAP), random_state=RANDOM_SEED)

df_benign = pd.read_csv(BENIGN_FILE)
df_benign = df_benign.sample(n=len(df_attack), random_state=RANDOM_SEED)

# Combine and shuffle
df = pd.concat([df_benign, df_attack], ignore_index=True).sample(frac=1, random_state=RANDOM_SEED)

X = df.drop(columns=["Label"])
y = df["Label"]

# 60/20/20 split with stratification
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=RANDOM_SEED, stratify=y)
X_val,   X_test, y_val,   y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp)

print(f"Attack: {ATTACK_FILE} | Total rows: {len(df)}")
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Attack: C:\Users\johfr\Dropbox\Skule\Data science\Bachelor\Dataset\Brute Force\combined_brute_force.csv | Total rows: 7238
Train: 4342 | Val: 1448 | Test: 1448


In [3]:
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_val   = X_val.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test  = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train_scaled)
y_train_t = torch.FloatTensor(y_train.values.copy())
X_val_t   = torch.FloatTensor(X_val_scaled)
y_val_t   = torch.FloatTensor(y_val.values.copy())
X_test_t  = torch.FloatTensor(X_test_scaled)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=256, shuffle=True)

In [4]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = X_train.shape[1]

class ANNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

model     = ANNModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

In [5]:
epochs = 50
patience          = 5
best_val_loss     = float("inf")
epochs_no_improve = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t.to(device))
        val_loss  = criterion(val_preds, y_val_t.to(device)).item()
        val_acc   = ((val_preds > 0.5) == y_val_t.to(device)).float().mean().item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

model.load_state_dict(torch.load(MODEL_PATH))

Epoch 1/50 | Loss: 0.6065 | Val Loss: 0.5839 | Val Acc: 0.7044
Epoch 2/50 | Loss: 0.5248 | Val Loss: 0.5040 | Val Acc: 0.7396
Epoch 3/50 | Loss: 0.4993 | Val Loss: 0.4758 | Val Acc: 0.7673
Epoch 4/50 | Loss: 0.4845 | Val Loss: 0.4652 | Val Acc: 0.7762
Epoch 5/50 | Loss: 0.4755 | Val Loss: 0.4698 | Val Acc: 0.7583
Epoch 6/50 | Loss: 0.4698 | Val Loss: 0.4540 | Val Acc: 0.7610
Epoch 7/50 | Loss: 0.4623 | Val Loss: 0.4466 | Val Acc: 0.7783
Epoch 8/50 | Loss: 0.4560 | Val Loss: 0.4475 | Val Acc: 0.7949
Epoch 9/50 | Loss: 0.4554 | Val Loss: 0.4473 | Val Acc: 0.7693
Epoch 10/50 | Loss: 0.4477 | Val Loss: 0.4354 | Val Acc: 0.7970
Epoch 11/50 | Loss: 0.4428 | Val Loss: 0.4374 | Val Acc: 0.8018
Epoch 12/50 | Loss: 0.4363 | Val Loss: 0.4271 | Val Acc: 0.7783
Epoch 13/50 | Loss: 0.4383 | Val Loss: 0.4205 | Val Acc: 0.8087
Epoch 14/50 | Loss: 0.4258 | Val Loss: 0.4173 | Val Acc: 0.8018
Epoch 15/50 | Loss: 0.4251 | Val Loss: 0.4150 | Val Acc: 0.8018
Epoch 16/50 | Loss: 0.4137 | Val Loss: 0.4128 | V

<All keys matched successfully>

In [6]:
model.eval()
with torch.no_grad():
    preds = (model(X_test_t.to(device)) > 0.5).cpu().numpy().astype(int)

print(classification_report(y_test, preds, target_names=["Benign", "Attack"]))
print(confusion_matrix(y_test, preds))

              precision    recall  f1-score   support

      Benign       0.88      0.72      0.79       724
      Attack       0.76      0.90      0.83       724

    accuracy                           0.81      1448
   macro avg       0.82      0.81      0.81      1448
weighted avg       0.82      0.81      0.81      1448

[[522 202]
 [ 70 654]]
